# T Gate

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from cmath import exp
from io import BytesIO
from math import pi, sqrt

import torch
from IPython.display import Image, Markdown, SVG, display
from matplotlib import pyplot as plt

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import PhaseCall
from flagquantum.compiler.registers.primitives.basis_state import basis_state_ir
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score
from flagquantum.simulation.unitary import get_unitary

## `TGate`
The T gate.

This is the fourth root of the Pauli Z gate. Quantum programs composed solely
of gates belonging to the Clifford group (like X, Z, Hadamard, CNOT, S) can be simulated
efficiently by a classical computer and therefore offer no quantum advantage. The Clifford
gates do not provide a universal quantum gateset. Adding T gives the universal Clifford+T gate set.

The unitary matrix of `cirq.T` is
$$
\begin{bmatrix}
    1 & 0 \\
    0 & e^{i \pi /4}
\end{bmatrix}
$$

#### Parameters
 - `is_adjoint`: If True, this bloq is $T^\dagger$ instead. 

#### Registers
 - `q`: The qubit 

#### References
 - [Universal Quantum Computation with ideal Clifford gates and noisy ancillas](https://arxiv.org/abs/quant-ph/0403025). Bravyi and Kitaev. 2004.
 - [Fast and efficient exact synthesis of single qubit unitaries generated by Clifford and T gates](https://arxiv.org/abs/1206.5236). Kliuchnikov et al. 2012.
 - [Universal Gate Set, Magic States, and costliness of the T gate](https://quantumcomputing.stackexchange.com/a/33358). Gidney. 2023.


In [ ]:
signature = Signature.build(q=1)

### Example Instances

In [ ]:
t_gate = PhaseCall("t")

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("T", signature, {})])))

### Call Graph

In [ ]:
graph = symbolic_call_graph(t_gate, {}, max_depth=1)
assert graph.leaf_counts == {t_gate: 1} and not graph.direct_calls
def title(_):
    return "T"
display(SVG(render_call_graph_svg(export_call_graph_dot(t_gate, graph.direct_calls, title=title))))
display(Markdown(format_call_graph_markdown(t_gate, graph.direct_calls, graph.leaf_counts, title=title)))
print(query_qec_gate_cost(t_gate))

### Properties & Relations

We can create the "magic state" T|+> which is used via teleportation to do non-Clifford operations in the surface code.

In [ ]:
dtype = torch.complex128
prepared = Circuit(1, dtype=dtype)
gates = (*basis_state_ir("x", 0, 0), Instruction("t", (0,)))
prepared.append(FieldOperation(1, gates))
expected = torch.tensor([1, exp(1j * pi / 4)], dtype=dtype) / sqrt(2)
torch.testing.assert_close(prepared.state().flatten(), expected, atol=2e-12, rtol=0)

layout = layout_register_score({"q": 1}, (), (
    RegisterStep("PlusState", outputs=("q",)),
    RegisterStep("T", ("q",), ("q",), "t"),
))

def wire_symbol(step, register, side):
    if register is None:
        return "Text", {"text": "", "fontsize": 10}
    if step.name == "PlusState":
        return "LarrowTextBox", {"text": "+"}
    return "TextBox", {"text": "T" if step.name == "T" else register}

fig, _ = draw_register_score(register_score_data(layout, wire_symbol=wire_symbol))
with BytesIO() as png:
    fig.savefig(png, format="png", bbox_inches="tight")
    display(Image(data=png.getvalue()))
plt.close(fig)
print(prepared.state().flatten() * sqrt(2))

### Check the exact inverse

Apply the adjoint to the prepared state to restore the plus amplitudes. Check the adjoint matrix and its one-T source cost separately from the H/T preparation stream.

In [ ]:
inverse = FieldOperation(1, (Instruction("tdg", (0,)),))
adjoint = Circuit(1, dtype=dtype).append(inverse)
expected_adjoint = torch.diag(torch.tensor([1, exp(-1j * pi / 4)], dtype=dtype))
torch.testing.assert_close(get_unitary(adjoint), expected_adjoint, atol=2e-12, rtol=0)
prepared.append(inverse)
torch.testing.assert_close(prepared.state().flatten(), torch.ones(2, dtype=dtype) / sqrt(2), atol=2e-12, rtol=0)
print("Preparation emitted", [gate.name for gate in gates])
print("Adjoint logical cost", query_qec_gate_cost(PhaseCall("t", adjoint=True)))